In [ ]:
#!pip install ipython
#!pip install langchain-openai

In [ ]:
# Imports

import requests
from typing import TypedDict, Dict, Any, List
import json

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI


In [ ]:
# CONFIG

BASE_URL = "http://localhost:8000"
ENDPOINT = "/api/chat"
OPENAI_KEY = "meinKey"

import os

os.environ["OPENAI_API_KEY"] = OPENAI_KEY

In [ ]:
# Send Chat message

def chat(message: str, api_key: str) -> str:
    response = requests.post(
        f"{BASE_URL}{ENDPOINT}",
        json={
            "message": message,
            "openAiKey": api_key
        },
        timeout=60,
    )

    response.raise_for_status()

    return response.json()["result"]["result"]

In [ ]:
# AGENT STATE

class TestAgentState(TypedDict):
    question: str
    expected: str
    actual: str

    semantic_score: float
    evaluation_reason: str

    status: str
    error: str

In [ ]:
# Define node functions


# NODE 1: INTERPRET INPUT
def interpret_test_case(state: TestAgentState):

    print("\n====================================")
    print("Running test")
    print("Question:", state["question"])
    print("====================================")

    return state


# NODE 2: RUN API TEST
def run_api_test(state: TestAgentState):

    try:
        actual = chat(state["question"], OPENAI_KEY)

        return {
            **state,
            "actual": actual,
            "error": ""
        }

    except Exception as e:

        return {
            **state,
            "actual": "",
            "error": str(e)
        }
    

# NODE 3: EVALUATE RESULT

judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

PASS_THRESHOLD = 0.85


def evaluate_result(state: TestAgentState):

    # Fehlerfall
    if state["error"]:

        return {
            **state,
            "status": "ERROR",
            "semantic_score": 0.0,
            "evaluation_reason": state["error"]
        }

    instructions = """
    Given an ACTUAL answer and an EXPECTED answer, determine whether
    the actual answer contains all of the information in the
    expected answer.

    Return ONLY valid JSON in this format:

    {
      "score": number between 0 and 1,
      "reason": "short explanation"
    }

    """

    #Evaluate the semantic similarity between
    #an ACTUAL answer and an EXPECTED answer.
    
    #Scoring rules:
    #1.0 = perfectly correct
    #0.8 = mostly correct
    #0.5 = partially correct
    #0.0 = completely wrong

    user_message = f"""
    ACTUAL ANSWER:
    {state["actual"]}

    EXPECTED ANSWER:
    {state["expected"]}
    """

    response = judge_llm.invoke([
        {
            "role": "system",
            "content": instructions
        },
        {
            "role": "user",
            "content": user_message
        }
    ])

    raw_response = response.content.strip()

    #print("Judge raw response:", raw_response)

    try:
        parsed = json.loads(raw_response)

        score = float(parsed["score"])
        reason = parsed["reason"]

    except Exception as e:

        score = 0.0
        reason = f"JSON parsing failed: {e}"

    status = "PASS" if score >= PASS_THRESHOLD else "FAIL"

    return {
        **state,
        "semantic_score": round(score, 3),
        "evaluation_reason": reason,
        "status": status
    }


# NODE 4: OUTPUT RESULT
def output_result(state: TestAgentState):

    print("Expected :", state["expected"])
    print("Actual   :", state["actual"])

    print("Semantic Score :", state["semantic_score"])
    print("Reason         :", state["evaluation_reason"])

    print("Status :", state["status"])

    if state["error"]:
        print("Error:", state["error"])

    print()

    return state

In [ ]:
# Build workflow

workflow = StateGraph(TestAgentState)

# Add nodes
workflow.add_node("interpret", interpret_test_case)
workflow.add_node("run_test", run_api_test)
workflow.add_node("evaluate", evaluate_result)
workflow.add_node("output", output_result)

# Define flow
workflow.add_edge(START, "interpret")
workflow.add_edge("interpret", "run_test")
workflow.add_edge("run_test", "evaluate")
workflow.add_edge("evaluate", "output")
workflow.add_edge("output", END)

# Compile graph
compiled_graph = workflow.compile()

In [ ]:
# Workflow visualisation

#from IPython.display import Image, display, Audio

#try:
#    display(Image(compiled_graph.get_graph().draw_mermaid_png()))
#except Exception:
#    # This requires some extra dependencies and is optional
#    pass

In [ ]:
# Execute Tests

def run_tests():

    passed = 0
    failed = 0

    print(f"\n=== Starting {len(TEST_CASES)} Tests ===\n")

    for test_case in TEST_CASES:

        result = compiled_graph.invoke({
            "question": test_case["question"],
            "expected": test_case["expected"],
            "actual": "",
            "similarity": 0.0,
            "status": "",
            "error": ""
        })

        if result["status"] == "PASS":
            passed += 1
        else:
            failed += 1

    print("=" * 50)
    print(f"Finished: {passed} passed, {failed} failed")
    print("=" * 50)

In [ ]:
# Run tests

# Test cases: (question, expected answer)
TEST_CASES = [
    {
        "question": "Where is the city of Coesfeld located?",
        "expected": "The city of Coesfeld is located in the district of Coesfeld, NRW, Germany.",
    },
    {
        "question": "Where is the district of Coesfeld located?",
        "expected": "The district of Coesfeld is located in the administrative district of Münster, NRW, Germany.",
    },
    {
        "question": "Where does the city of Coesfeld lie?",
        "expected": "The city of Coesfeld lies in the district of Coesfeld, NRW, Germany.",
    },
    {
        "question": "Where does the district of Coesfeld lie?",
        "expected": "The district of Coesfeld lies in the administrative district of Münster, NRW, Germany.",
    },
    {
        "question": "Where is the city of Coesfeld?",
        "expected": "The city of Coesfeld is in the district of Coesfeld, NRW, Germany.",
    },
    {
        "question": "Where is the district of Coesfeld?",
        "expected": "The district of Coesfeld is in the administrative district of Münster, NRW, Germany.",
    },

]


run_tests()